# 02 · Train DistilBERT on SQuAD 2.0

**Task T3** in [TASKS.md](../TASKS.md): a smoke test. Run this top to bottom in Google Colab (Runtime → Change runtime type → **T4 GPU**).

A smoke test is a very short training run. It doesn't produce a useful model. It proves that every part of training works (data, model, loss, checkpoints) before we spend hours on the full run in T6.

What this notebook does:
1. Trains DistilBERT for 50 steps on 1% of the training data
2. Plots the training loss, which should go down
3. Checks that checkpoints and the final model were saved
4. Loads the saved model and asks it one question

All the code is in `src/train.py`. All the settings are in `configs/distilbert.yaml`, with the smoke-test changes in `configs/smoke.yaml`.

## 0. Setup
Clones the repo and installs the requirements in Colab. Safe to run more than once.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Davidfaleye360/AI-Question-Answering-System.git"
REPO_DIR = "/content/AI-Question-Answering-System"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import transformers, datasets, torch

print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError("transformers 5.x is still loaded. Use Runtime -> Restart session, then run the setup cell again.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then run the setup cell again.")
    print("The smoke test still works without one, just more slowly.")

## 1. The settings
`configs/smoke.yaml` lists only what the smoke test changes. Everything else (learning rate, batch size, window size) is the same as the full run, so the smoke test exercises the real settings.

In [ ]:
from src.config import load_config

full = load_config("configs/distilbert.yaml")
cfg = load_config("configs/distilbert.yaml", overrides="configs/smoke.yaml")

print(f"Data used:      {cfg.data.train_fraction:.0%} of the training set (full run: {full.data.train_fraction:.0%})")
print(f"Training steps: {cfg.training.max_steps} (full run: {full.training.num_epochs} epochs)")
print(f"Batch size:     {cfg.training.batch_size} windows per step")
print(f"Learning rate:  {cfg.training.learning_rate}")
print(f"Loss recorded every {cfg.training.logging_steps} steps, checkpoint every {cfg.training.save_steps} steps")
print(f"Saving to:      {cfg.training.output_dir}")

## 2. Train for 50 steps

**What to expect.** The model has to pick the answer's start token and end token out of 384 positions. Before training it is guessing, and the loss for a blind guess among 384 options is about ln(384) ≈ 5.95. As it learns, the loss should fall well below that.

You'll also see a warning that some weights were "newly initialized". That's expected: the pretrained DistilBERT has no question-answering layer, so a new one is added and trained here.

`--fresh` starts from step 0 every time. Without it, the script continues from the newest checkpoint in the output folder, which is what we want for the long run in T6.

In [ ]:
!{sys.executable} -m src.train --config configs/distilbert.yaml --overrides configs/smoke.yaml --fresh

The last line must say **RESULT: PASS**: the loss went down, checkpoints were saved, and the final model was saved.

## 3. The loss curve

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

summary = json.load(open(f"{cfg.training.output_dir}/train_summary.json"))
losses = pd.DataFrame(summary["losses"], columns=["step", "loss"]).set_index("step")

ax = losses["loss"].plot(marker="o", color="#2a78d6", figsize=(7, 3.5))
ax.set_title(f"Training loss, smoke test ({summary['steps']} steps on {summary['device']})")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"First recorded loss: {losses['loss'].iloc[0]:.3f} | last: {losses['loss'].iloc[-1]:.3f} "
      f"| training time: {summary['runtime_seconds']:.0f}s")
print("Loss went down:", summary["loss_went_down"], "| RESULT:", "PASS" if summary["passed"] else "FAIL")

Each point is the average loss over the previous few steps. It jumps around from batch to batch, so look at the trend, not single points.

## 4. What was saved
Each `checkpoint-N` folder holds the model weights plus the optimizer state at step N, which is what lets a run continue after a disconnect. The files at the top level are the final model and tokenizer.

In [ ]:
from pathlib import Path

out = Path(cfg.training.output_dir)
for path in sorted(out.iterdir()):
    if path.is_dir():
        size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
        print(f"{path.name + '/':28} {size / 1e6:8.1f} MB")
    else:
        print(f"{path.name:28} {path.stat().st_size / 1e6:8.1f} MB")

## 5. Load the saved model
This checks that the saved files really are a usable model. After only 50 steps the answers are usually wrong or empty: the point here is that it loads and runs, not that it's right.

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering", model=cfg.training.output_dir, device=0 if torch.cuda.is_available() else -1)
qa(question="When did Curie win the Chemistry prize?",
   context="Marie Curie won the Nobel Prize in Physics in 1903 and in Chemistry in 1911.",
   handle_impossible_answer=True)

## Next
- If section 2 printed **RESULT: PASS**, tick **T3** in TASKS.md and note the first and last loss.
- The full training run (**T6**) will be added to this notebook: same script without `--overrides`, saving checkpoints to Google Drive.